# Submission Inspection

Goal: understand the **exact** schema Kaggle expects for `is_anomaly` predictions so our any future model submissions don't get rejected or silently scored as 0.

## What is parquet?

Parquet is a **columnar, compressed binary file format** — think of it as a faster, smaller, typed version of CSV.

- **Columnar**: data is stored column-by-column, not row-by-row → reading only a few columns is very fast.
- **Typed**: every column has a fixed dtype (int32, float64, bool, …) baked into the file. CSVs have no dtypes; parquet does.
- **Compressed**: typically 5–10× smaller than the equivalent CSV.
- **Readable with pandas**: `pd.read_parquet(path)` — that's it. No config.

Why Kaggle uses it here: the train file is 14.7 M rows × 89 columns. In CSV that would be ~5 GB; parquet keeps it under 1 GB.

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

# Project-relative paths (notebook runs from notebooks/)
PROJECT_ROOT = Path('..').resolve()
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
SUBMISSIONS_DIR = PROJECT_ROOT / 'submissions'

print('Project root            :', PROJECT_ROOT)
print('submissions/ contents   :', [p.name for p in SUBMISSIONS_DIR.glob('*.parquet')])

Project root            : /Users/helena.schulz.ext/code/alexfederolf/sentinel
submissions/ contents   : ['baseline_pca.parquet', 'baseline_ensemble.parquet', 'pca_full.parquet', 'cnn_ae.parquet', 'baseline_iforest.parquet', 'baseline_lstm_ae.parquet']


## All submissions

In [ ]:
SUB = SUBMISSIONS_DIR / 'baseline_lstm_ae.parquet'
sub = pd.read_parquet(SUB)

def schema_overview(path: Path, template: pd.DataFrame) -> dict:
    """Eine Zeile pro Submission — nur das Wesentliche."""
    df = pd.read_parquet(path)
    n = len(df)
    if 'is_anomaly' in df.columns:
        vc = df['is_anomaly'].value_counts(dropna=False)
        count_0 = int(vc.get(0, 0))
        count_1 = int(vc.get(1, 0))
        rate = count_1 / n if n else 0.0
    else:
        count_0 = count_1 = -1
        rate = -1.0
    return {
        'file'           : path.name,
        'rows_match'     : n == len(template),
        'dtype_match'    : df.dtypes.equals(template.dtypes) if list(df.columns) == list(template.columns) else False,
        'dtype'          : str(df['is_anomaly'].dtype) if 'is_anomaly' in df.columns else 'MISSING',
        'count_0'        : count_0,
        'count_1'        : count_1,
        'anomaly_rate'   : round(rate, 4),
    }


overview = pd.DataFrame(
    [schema_overview(p, sub) for p in sorted(SUBMISSIONS_DIR.glob('*.parquet'))]
)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
overview

NameError: name 'sub' is not defined

### 5B. Detail-Analyse einer einzelnen Submission

Wähle unten die Datei, die du genauer anschauen willst. Der Rest der Zellen verwendet automatisch die Auswahl.

Was wir uns anschauen:
- **Schema** im Vergleich zum Template
- **Head / Tail** der Datei
- **Anzahl 0 vs 1** und Anomalie-Rate
- **Event-Struktur**: wie viele zusammenhängende 1-Segmente gibt es? Wie lang sind sie?
- **Verteilung der Event-Längen** (wichtig für F0.5: kurze, kompakte Detektionen werden belohnt)

#### Event-Struktur

F0.5 wird **event-weise** gewertet: jedes zusammenhängende 1-Segment zählt als ein Event. Ein Modell mit 100 winzigen Events bei 1 % Anomalie-Rate ist meistens besser als eines mit 5 riesigen Segmenten — solange die kleinen nicht zu viele False Positives werfen.

Hier zählen wir, wie viele zusammenhängende 1-Blöcke es gibt und wie lang sie sind.

In [51]:
# Hier umschalten:
SELECTED_FILE = 'baseline_lstm_ae.parquet'
# Optionen:  baseline_iforest.parquet | baseline_pca.parquet | baseline_lstm_ae.parquet | pca_full.parquet | dummy_all_zeros.parquet

selected_path = SUBMISSIONS_DIR / SELECTED_FILE
assert selected_path.exists(), f'Datei nicht gefunden: {selected_path}'
sel = pd.read_parquet(selected_path)

print(f'Datei            : {SELECTED_FILE}')
print(f'Shape            : {sel.shape}')
print(f'Spalten          : {list(sel.columns)}')
print(f'Dtypes           :')
print(sel.dtypes)
print(f'\nis_anomaly dtype : {sel["is_anomaly"].dtype}')
print(f'Template dtype   : {sub["is_anomaly"].dtype}')
print(f'Dtype stimmt     : {sel["is_anomaly"].dtype == sub["is_anomaly"].dtype}')

Datei            : baseline_lstm_ae.parquet
Shape            : (521280, 2)
Spalten          : ['id', 'is_anomaly']
Dtypes           :
id            int64
is_anomaly    uint8
dtype: object

is_anomaly dtype : uint8
Template dtype   : uint8
Dtype stimmt     : True


## 1. Load submission

`pd.read_parquet` works on any parquet file. If you get a missing-engine error, install `pyarrow` (`pip install pyarrow`).

In [45]:
SUB = SUBMISSIONS_DIR / 'baseline_lstm_ae.parquet'

sub = pd.read_parquet(SUB)

print('Shape           :', sub.shape)
print('Columns         :', list(sub.columns))
print('Dtypes          :')
print(sub.dtypes)
print('\nMemory footprint:', f'{sub.memory_usage(deep=True).sum() / 1e6:.2f} MB')

Shape           : (521280, 2)
Columns         : ['id', 'is_anomaly']
Dtypes          :
id            int64
is_anomaly    uint8
dtype: object

Memory footprint: 4.69 MB


In [46]:
# First and last 5 rows — verify id is contiguous and where it starts
print('Head:')
print(sub.head())
print('\nTail:')
print(sub.tail())

Head:
         id  is_anomaly
0  14728321           1
1  14728322           1
2  14728323           1
3  14728324           1
4  14728325           1

Tail:
              id  is_anomaly
521275  15249596           1
521276  15249597           1
521277  15249598           1
521278  15249599           1
521279  15249600           1


## 2. Cross-check row count and id range against test.parquet

According to the challenge notes:
- `test.parquet` = 521,280 rows
- `id` is continuous across train (0 … 14,728,320) and test (14,728,321 … 15,249,600)

We verify this from the actual files — never trust notes blindly.

In [47]:
# Read only the `id` column from test.parquet — fast, because parquet is columnar
test_ids = pd.read_parquet(TEST, columns=['id'])

print('test.parquet rows             :', len(test_ids))
print('test id range                 :', test_ids['id'].min(), '\u2026', test_ids['id'].max())
print('sample_submission rows        :', len(sub))
print('sample_submission id range    :', sub['id'].min(), '\u2026', sub['id'].max())

assert len(sub) == len(test_ids), 'Row count mismatch between submission and test!'
assert (sub['id'].values == test_ids['id'].values).all(), 'id columns do not match row-by-row!'
print('\nOK \u2014 submission ids align perfectly with test ids.')

test.parquet rows             : 521280
test id range                 : 14728321 … 15249600
sample_submission rows        : 521280
sample_submission id range    : 14728321 … 15249600

OK — submission ids align perfectly with test ids.


## 3. Inspect the `is_anomaly` column

We need to know:
- **dtype** (int8 / int64 / bool / float?) — Kaggle is usually strict about this
- **value set** (only {0, 1}? or are floats allowed?)
- whether there are any NaNs

In [48]:
col = 'is_anomaly'

print(f'dtype                : {sub[col].dtype}')
print(f'unique values        : {sorted(sub[col].unique().tolist())}')
print(f'value counts         :\n{sub[col].value_counts(dropna=False)}')
print(f'NaN count            : {sub[col].isna().sum()}')

dtype                : uint8
unique values        : [1]
value counts         :
is_anomaly
1    521280
Name: count, dtype: int64
NaN count            : 0


## 4. Inspect the raw parquet schema (not just the pandas view)

Pandas sometimes upcasts dtypes on read. To see what is **actually** stored in the parquet file, use `pyarrow` directly. This is the schema Kaggle's grader sees.

In [49]:
import pyarrow.parquet as pq

pq_file = pq.ParquetFile(SAMPLE_SUB)
print('Parquet schema (raw):')
print(pq_file.schema_arrow)
print('\nNumber of row groups:', pq_file.num_row_groups)
print('Total rows          :', pq_file.metadata.num_rows)

Parquet schema (raw):
id: int32
is_anomaly: uint8
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 482

Number of row groups: 1
Total rows          : 521280


In [52]:
print('Head:')
print(sel.head())
print('\nTail:')
print(sel.tail())

Head:
         id  is_anomaly
0  14728321           1
1  14728322           1
2  14728323           1
3  14728324           1
4  14728325           1

Tail:
              id  is_anomaly
521275  15249596           1
521276  15249597           1
521277  15249598           1
521278  15249599           1
521279  15249600           1


In [53]:
vc = sel['is_anomaly'].value_counts(dropna=False).sort_index()
n = len(sel)

print(f'Gesamtzeilen     : {n:>10,}')
for val, cnt in vc.items():
    print(f'  is_anomaly = {val}: {cnt:>10,}  ({cnt / n * 100:6.2f} %)')
print(f'NaN              : {sel["is_anomaly"].isna().sum():>10,}')

Gesamtzeilen     :    521,280
  is_anomaly = 1:    521,280  (100.00 %)
NaN              :          0


In [54]:
def segment_lengths(binary: np.ndarray) -> np.ndarray:
    """Gibt die Längen aller zusammenhängenden 1-Segmente zurück."""
    if binary.sum() == 0:
        return np.array([], dtype=int)
    # Finde Start- und End-Indizes von 1-Runs
    diff = np.diff(np.concatenate(([0], binary, [0])))
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0]
    return ends - starts


segs = segment_lengths(sel['is_anomaly'].values.astype(np.int8))

if len(segs) == 0:
    print('Keine Anomalien in dieser Submission.')
else:
    print(f'Anzahl Events          : {len(segs):>8,}')
    print(f'Kürzestes Event        : {segs.min():>8,}  Samples')
    print(f'Längstes Event         : {segs.max():>8,}  Samples')
    print(f'Median Event-Länge     : {int(np.median(segs)):>8,}  Samples')
    print(f'Mittlere Event-Länge   : {segs.mean():>8,.1f}  Samples')
    print(f'Summe aller 1-Samples  : {segs.sum():>8,}  (= count_1)')

    # Verteilung in Buckets
    print('\nLängen-Verteilung:')
    bins = [0, 1, 5, 20, 100, 1000, segs.max() + 1]
    labels = ['1', '2-5', '6-20', '21-100', '101-1000', '>1000']
    hist, _ = np.histogram(segs, bins=bins)
    for lbl, cnt in zip(labels, hist):
        print(f'  {lbl:>10s}  Samples : {cnt:>6,} Events')

Anzahl Events          :        1
Kürzestes Event        :  521,280  Samples
Längstes Event         :  521,280  Samples
Median Event-Länge     :  521,280  Samples
Mittlere Event-Länge   : 521,280.0  Samples
Summe aller 1-Samples  :  521,280  (= count_1)

Längen-Verteilung:
           1  Samples :      0 Events
         2-5  Samples :      0 Events
        6-20  Samples :      0 Events
      21-100  Samples :      0 Events
    101-1000  Samples :      0 Events
       >1000  Samples :      1 Events


## 6. Build a reusable `make_submission()` helper

We start with the sample submission itself (guaranteed correct shape + dtypes) and just overwrite `is_anomaly` with our predictions. This is the safest pattern.

In [55]:
def make_submission(predictions: np.ndarray, template_path: Path = SAMPLE_SUB) -> pd.DataFrame:
    """Return a submission dataframe with the exact schema of the Kaggle template.

    Parameters
    ----------
    predictions : 1-D array of 0/1 values, length must match len(test).
    template_path : path to sample_submission.parquet (source of truth for schema).
    """
    template = pd.read_parquet(template_path)
    if len(predictions) != len(template):
        raise ValueError(
            f'predictions length {len(predictions)} != template length {len(template)}'
        )
    out = template.copy()
    # Cast predictions to the template's exact dtype
    out['is_anomaly'] = np.asarray(predictions).astype(template['is_anomaly'].dtype)
    return out


# Dummy: predict no anomalies anywhere
dummy_preds = np.zeros(len(sub), dtype=np.int8)
dummy_sub = make_submission(dummy_preds)

print('Dummy submission shape :', dummy_sub.shape)
print('Dummy dtypes           :')
print(dummy_sub.dtypes)
print('\nHead:')
print(dummy_sub.head())

Dummy submission shape : (521280, 2)
Dummy dtypes           :
id            int32
is_anomaly    uint8
dtype: object

Head:
         id  is_anomaly
0  14728321           0
1  14728322           0
2  14728323           0
3  14728324           0
4  14728325           0


## 7. Save the dummy submission and reload to verify

Round-trip test: write → read → compare schema. If the reloaded file's dtypes drift, Kaggle may reject it.

In [56]:
dummy_path = SUBMISSIONS_DIR / 'dummy_all_zeros.parquet'
dummy_sub.to_parquet(dummy_path, index=False)

print(f'Saved to: {dummy_path}')
print(f'File size: {dummy_path.stat().st_size / 1024:.1f} KB')

# Reload and compare
reloaded = pd.read_parquet(dummy_path)

print('\nReloaded shape   :', reloaded.shape)
print('Reloaded dtypes  :')
print(reloaded.dtypes)

schema_ok = (
    list(reloaded.columns) == list(sub.columns)
    and (reloaded.dtypes == sub.dtypes).all()
    and len(reloaded) == len(sub)
)
print('\nSchema matches sample_submission exactly :', schema_ok)

Saved to: /Users/helena.schulz.ext/code/alexfederolf/sentinel/submissions/dummy_all_zeros.parquet
File size: 2614.5 KB

Reloaded shape   : (521280, 2)
Reloaded dtypes  :
id            int32
is_anomaly    uint8
dtype: object

Schema matches sample_submission exactly : False


## 8. Summary — rules for every future submission

Before uploading to Kaggle, a submission must:

1. Have exactly the same **row count** as `sample_submission.parquet`.
2. Have exactly the same **columns** in the same **order**.
3. Have the same **dtype** for `is_anomaly` as the template (printed above).
4. Have `id` aligned row-for-row with `test.parquet`.
5. Contain **no NaNs** in `is_anomaly`.

The `make_submission(predictions)` helper above enforces all five as long as you feed it a 1-D array of predictions in the same order as `test.parquet`.

### Next steps for the CNN-AE pipeline

- Run CNN-AE inference over `test.parquet` → per-sample reconstruction error array of length 521,280.
- Apply threshold + post-processing (smoothing, min-duration filter, event merging) → binary `is_anomaly` array.
- `make_submission(binary_preds).to_parquet('../submissions/cnn_ae_v1.parquet', index=False)`
- Upload to Kaggle.